# 🚀 SIRCCD - Entrenamiento YOLOv8 en Google Colab

Este notebook entrena un modelo YOLOv8 para detección de residuos usando el dataset SIRCCD.

**GPU Recomendada**: T4 (15 GB VRAM)

**Tiempo estimado**: 6-8 horas (100 epochs)

---

## 📋 Checklist Inicial

- [ ] Runtime > Change runtime type > GPU > T4
- [ ] Tener credenciales de MinIO listas
- [ ] Google Drive conectado (para guardar modelo)

## 🔧 1. Setup Inicial

In [ ]:
# Verificar GPU disponible
!nvidia-smi

In [ ]:
# Instalar dependencias
!pip install -q ultralytics minio python-dotenv pillow

In [ ]:
# Montar Google Drive (para guardar modelo)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Mantener sesión activa
from google.colab import output
output.enable_keepalive()

## 🗄️ 2. Configurar MinIO y Descargar Dataset

In [ ]:
# Configurar credenciales MinIO
import os

# ⚠️ REEMPLAZA CON TUS CREDENCIALES
os.environ['MINIO_ENDPOINT'] = 'tu-servidor:9000'  # Ej: 192.168.1.100:9000
os.environ['MINIO_ACCESS_KEY'] = 'tu-access-key'
os.environ['MINIO_SECRET_KEY'] = 'tu-secret-key'
os.environ['MINIO_USE_SSL'] = 'false'

BUCKET_NAME = 'sirccd-datasets'
DATASET_VERSION = 'v1.0.0'

In [ ]:
# Descargar dataset desde MinIO
from minio import Minio
from tqdm import tqdm
import os

# Conectar a MinIO
minio_client = Minio(
    os.environ['MINIO_ENDPOINT'],
    access_key=os.environ['MINIO_ACCESS_KEY'],
    secret_key=os.environ['MINIO_SECRET_KEY'],
    secure=(os.environ.get('MINIO_USE_SSL', 'false').lower() == 'true')
)

print("✅ Conectado a MinIO")

# Crear directorios
os.makedirs('/content/datasets/sirccd/images', exist_ok=True)
os.makedirs('/content/datasets/sirccd/labels', exist_ok=True)

# Descargar imágenes
print("📥 Descargando imágenes...")
image_objects = list(minio_client.list_objects(
    BUCKET_NAME,
    prefix=f'{DATASET_VERSION}/images/',
    recursive=True
))

for obj in tqdm(image_objects, desc="Imágenes"):
    local_path = f"/content/datasets/sirccd/images/{os.path.basename(obj.object_name)}"
    minio_client.fget_object(BUCKET_NAME, obj.object_name, local_path)

print(f"✅ Descargadas {len(image_objects)} imágenes")

# Descargar labels
print("📥 Descargando labels YOLO...")
label_objects = list(minio_client.list_objects(
    BUCKET_NAME,
    prefix=f'{DATASET_VERSION}/labels_yolo/',
    recursive=True
))

for obj in tqdm(label_objects, desc="Labels"):
    if obj.object_name.endswith('.txt'):
        local_path = f"/content/datasets/sirccd/labels/{os.path.basename(obj.object_name)}"
        minio_client.fget_object(BUCKET_NAME, obj.object_name, local_path)

print(f"✅ Descargadas {len(label_objects)} labels")

## 📝 3. Crear Archivo de Configuración YOLO

In [ ]:
# data.yaml para entrenamiento
data_yaml = """
# SIRCCD Dataset Configuration
path: /content/datasets/sirccd
train: images  # carpeta de imágenes de entrenamiento
val: images    # carpeta de imágenes de validación (YOLO hace split automático)

# Classes (actualiza según tus clases)
names:
  0: residuo
  1: contenedor
  2: vehiculo

# Number of classes
nc: 3
"""

with open('/content/datasets/sirccd/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml creado")
!cat /content/datasets/sirccd/data.yaml

## 🎯 4. Entrenar Modelo YOLOv8

In [ ]:
from ultralytics import YOLO

# Cargar modelo pre-entrenado
model = YOLO('yolov8n.pt')  # nano - más rápido
# model = YOLO('yolov8s.pt')  # small - mejor precisión
# model = YOLO('yolov8m.pt')  # medium - requiere más VRAM

print("✅ Modelo cargado")

In [ ]:
# Configuración de entrenamiento
EPOCHS = 100
BATCH_SIZE = 16  # Ajusta según GPU (T4: 16-32, V100: 32-64)
IMG_SIZE = 640
PROJECT_NAME = 'sirccd-training'
RUN_NAME = 'baseline-yolov8n'

# Entrenar
results = model.train(
    data='/content/datasets/sirccd/data.yaml',
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,  # GPU
    amp=True,  # Mixed precision (2x más rápido)
    cache=True,  # Cache imágenes en RAM
    project=PROJECT_NAME,
    name=RUN_NAME,
    save_period=10,  # Guardar checkpoint cada 10 epochs
    patience=50,  # Early stopping si no mejora en 50 epochs
    verbose=True,
    
    # Data augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.0
)

print("\n🎉 Entrenamiento completado!")

## 📊 5. Evaluar Modelo

In [ ]:
# Validar en dataset de validación
metrics = model.val()

print("\n📊 Métricas de Validación:")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Visualizar predicciones de ejemplo
from IPython.display import Image, display
import glob

# Tomar algunas imágenes de ejemplo
test_images = glob.glob('/content/datasets/sirccd/images/*.jpg')[:5]

for img_path in test_images:
    results = model.predict(img_path, save=True, conf=0.25)
    
print("\n📸 Predicciones guardadas en:")
!ls -lh runs/detect/predict*/

# Mostrar una predicción
pred_images = glob.glob(f'{PROJECT_NAME}/{RUN_NAME}/val_batch*.jpg')
if pred_images:
    display(Image(filename=pred_images[0]))

## 💾 6. Guardar Modelo en Google Drive

In [ ]:
import shutil
from datetime import datetime

# Crear carpeta en Drive
drive_folder = '/content/drive/MyDrive/SIRCCD_Models'
os.makedirs(drive_folder, exist_ok=True)

# Timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_folder = f'{drive_folder}/{RUN_NAME}_{timestamp}'

# Copiar resultados completos
shutil.copytree(
    f'{PROJECT_NAME}/{RUN_NAME}',
    model_folder
)

print(f"✅ Modelo guardado en Google Drive:")
print(f"   {model_folder}")
print(f"\n📁 Archivos incluidos:")
print(f"   - weights/best.pt    (mejor modelo)")
print(f"   - weights/last.pt    (último epoch)")
print(f"   - results.csv        (métricas)")
print(f"   - confusion_matrix.png")
print(f"   - PR_curve.png")
print(f"   - F1_curve.png")

## 📤 7. (Opcional) Subir Modelo a MinIO

In [ ]:
# Subir best.pt a MinIO
import os

best_model_path = f'{PROJECT_NAME}/{RUN_NAME}/weights/best.pt'
minio_object_name = f'models/{RUN_NAME}_{timestamp}/best.pt'

try:
    minio_client.fput_object(
        BUCKET_NAME,
        minio_object_name,
        best_model_path
    )
    print(f"✅ Modelo subido a MinIO: {minio_object_name}")
except Exception as e:
    print(f"❌ Error subiendo a MinIO: {e}")
    print("   (No te preocupes, el modelo está en Google Drive)")

## 🔔 8. Notificación de Finalización

In [ ]:
# Resumen final
import json

summary = {
    'model': RUN_NAME,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'mAP50': float(metrics.box.map50),
    'mAP50-95': float(metrics.box.map),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'timestamp': timestamp,
    'drive_path': model_folder
}

print("\n" + "="*60)
print("🎉 ENTRENAMIENTO COMPLETADO")
print("="*60)
print(json.dumps(summary, indent=2))
print("="*60)

# Guardar resumen
with open(f'{model_folder}/training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Descarga los archivos desde Google Drive")
print("   O continúa experimentando en este notebook")

---

## 🚀 Próximos Pasos

1. **Descargar modelo desde Google Drive** a tu PC
2. **Evaluar en casos reales** con nuevas imágenes
3. **Fine-tuning**:
   - Probar YOLOv8s/m para mejor precisión
   - Ajustar data augmentation
   - Entrenar más epochs
4. **Optimizar para producción**:
   - Exportar a ONNX: `model.export(format='onnx')`
   - Exportar a TensorRT para Jetson Nano
5. **Integrar con backend** del proyecto SIRCCD

---

**Documentación**:
- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com)
- [Google Colab Tips](https://colab.research.google.com/notebooks/)

**Soporte**: Ver `ml/docs/CLOUD_TRAINING.md` en el repositorio